In [ ]:
# 에이전트는 LLM이 환경에서 받은 피드백을 바탕으로 반복적으로 도구를 호출하며 문제를 해결하는 구조임.
# 미리 단계수를 정할 수 없는 개방형 문제에 적합하며, LLM이 스스로 다음 행동을 결정함.

# 아래 에이전트는 코드 LLM이 환경 피드백을 받고(도구 결과 등) 필요하면 도구를 호출하면서 반복 루프로 행동을 이어가는 전형적인 구조임.
# 이 예제에서는 덧셈, 곱셈, 나눗셈 도구를 정의하고, LLM이 필요 시 해당 도구를 호출해 연산을 수행하도록 설계함.
# 에이전트의 핵심은 LLM에 적절한 도구 세트를 달아 주고, 매 턴마다 도구를 쓸지 말지 판단 -> 실행 -> 관찰 값을 다시 LLM에 다시 돌려주는 루프를 만드는 것임.

# 01. 도구 정의
######################################################

# 이 단계에서는 LLM이 호출할 수 있는 함수를 정의함. @tool 데코레이터를 사용하면 랭체인이 해당 함수를 LLM이 이해할 수있는 형태로 등록함.
# 예제에서는 multiply, add, divide 의 세가지 산술 연산 함수를 만들었음.
# 각 함수는 입력 매개변수와 반환 타입이 명시돼 있어 LLM이 올바른 형식으로 출력할 수 있음.

from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END

load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

@tool
def multiply(a: int, b: int):
    """두수를 곱한다."""
    return a * b

@tool
def add(a: int, b: int):
    """두수를 더한다."""
    return a + b

@tool
def divide(a: int, b: int):
    """두수를 나눈다"""
    return a / b

# 아래 코드는 정의한 도구를 LLM과 연결하는 단계임.
# 먼저 모든 도구를 리스트로 묶고, 도구 이름을 키로 하는 딕셔너리를 생성해 빠르게 접근할수 있도록 함.
# 그런 다음, llm.bind_tools() 를 호출해 LLM 의 해당 도구들을 사용할 수 잇게 바인딩함. 이 과정을 거치면 LLM은 대화 도중 필요할 때 적절한 도구를 호출할 수 있음.
tools = [add, multiply, divide]
tools_by_name = {tools.name: tool for tool in tools}
llm_with_tools =llm.bind_tools(tools)

In [ ]:
# 02. 노드 정의
######################################################

# llm_call 함수는 LLM이 도구 호출 여부를 결정하는 역할을 함.
# 현재까지의 대화 내용을 기반으로 LLM 이 판단을 내리며, 도구 호출이 필요하면 tool_calls 형식으로 요청을 생성함.
# 이때 SystemMessage 를 사용해 LLM에게 당신은 산술연산을 하는 어시스턴트라는 역할 지침을 부여해 목적에 맞는 답변과 행동을 하도록 유도함

# MessagesState 는 랭그래프가 제공하는 기본, 상태 탬플릿 임. 이름 그대로 그래프가 실행되는 동안 오가는 메시지 대화 이력(messages)를 저장하는 구조를 이미 포함하고 잇음.
# 따라서 '메시지 기반으로만 상태를 관리하는 간단한 그래프'를 만들 때는 별도의 상태 타입을 직접 정의하지 않고 MessagesState를 그대로 사용하는 것이 가장 간편.

from langgraph.graph import MessagesState
from langchain_core import SystemMessage, HumanMessage, ToolMessage

# LLM 호출 노드
def llm_call(state: dict):
    """LLM 도구 호출 여부를 결정"""
    return {
        "messages": [
            llm_with_tools.invoke(
                [SystemMessage(content="당신은 주어진 입력값으로 산술 연산을 수행하는 유용한 어시스턴트입니다.")] + state["messages"]
            )
        ]
    }

# 아래 tool_node 함수는 LLM이 요청한 도구를 실제로 실행하는 단계임. LLM이 tool_calls 를 통해 전달한 도구 이름과 인자를 기반으로 미리 정의한 파이썬 함수를 찾아 실행함.
# 실행 결과는 ToolMassage 형태로 LLM에게 다시 전달되며, LLM은 이 결과를 바탕으로 후속 결정을 내릴 수 있음.

# 도구 실행 노드
def tool_node(state:dict):
    """LLM이 요청한 도구를 생성"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id= tool_call["id"]))

    return {"messages": result}

# should_continue 함수는 LLM이 도구 호출을 계속할지 종료해야 할지를 판단함.
# 마지막 LLM 응답에 tool_calls 가 포함돼 있으면 도구 실행 단계로 되돌아 가고, 그렇지 않으면 대화를 종료함. 이를 통해 불필요한 반복을 방지하면서 필요한 경우에는 연속적인 작업 수행이 가능함.

# 반복 여부 판단.
def should_continue(state: MessagesState):
    """도구 호출이 있으면 계속, 없으면 종료"""
    if state["messages"][-1].tool_calls:
        return "Action"
    return END

In [ ]:
# 03. 워크플로우 구성
######################################################

# 워크플로우는 StateGraph 를 사용해 구성.
agent_builder = StateGraph(MessagesState)
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("environment", tool_node)

agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "Action": "environment", END : END
    }
)
agent_builder.add_edge("environment", "llm_call")

agent = agent_builder.compile()

# 실행.
messages = [HumanMessage(content="3과 4를 더하세요")]
messages = agent.invoke({"messages": messages})

for m in messages["messages"]:
    m.pretty_print()

# 예제를 실행하면 "3과 4를 더하세요"라는 요청이 LLM에 전달됨.
# LLM은 add 도구 호출을 결정하고 tool_node 가 이를 실행한 결과 7을 반환함. LLM은 이 결과를 받아 최종적으로 사용자에게 7입니다. 같은 응답을 생성함.
# 사용자가 복잡한 작업을 요청해도 이 과정은 동일한 방식으로 확장할 수 있음.